In [110]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool, InjectedToolArg
from langchain_core.messages import HumanMessage, ToolMessage
from typing import Annotated
import requests
import os

In [ ]:
# tool create
from typing import Annotated
from langchain_core.tools import InjectedToolArg

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/86f41d53b364201652bc034d/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int,conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """Convert the base currency value using the supplied conversion rate."""

    return base_currency_value * conversion_rate

In [89]:
get_conversion_factor.args

{'base_currency': {'title': 'Base Currency', 'type': 'string'},
 'target_currency': {'title': 'Target Currency', 'type': 'string'}}

In [90]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [91]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

92.753

In [92]:
convert.invoke({'base_currency_value':10, 'conversion_rate':92.753})

927.53

In [93]:
#now we will bind tool with llm

llm = ChatGroq(model = "openai/gpt-oss-120b")

In [94]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [101]:
#tool calling 
messages = [
    HumanMessage(
        "Find the conversion factor from INR to USD "
        "using get_conversion_factor."
    )
]

In [102]:
messages

[HumanMessage(content='Find the conversion factor from INR to USD using get_conversion_factor.', additional_kwargs={}, response_metadata={})]

In [106]:
ai_message = llm_with_tools.invoke(messages)

In [104]:
messages.append(ai_message)

In [109]:
tool_call = ai_message.tool_calls[0]

tool_result = get_conversion_factor.invoke(tool_call)

conversion_rate = tool_result.content

print(conversion_rate)

0.01078


In [100]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)

TypeError: 'float' object is not subscriptable

In [ ]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': "We need conversion factor INR to USD. Use get_conversion_factor. Then convert 10 INR using conversion factor. We'll call function.", 'tool_calls': [{'id': 'fc_e8cafbcc-6d3a-4ca0-8f53-637dfa8b8860', 'function': {'arguments': '{"base_currency":"INR","target_currency":"USD"}', 'name': 'get_conversion_factor'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 201, 'total_tokens': 266, 'completion_time': 0.141628227, 'completion_tokens_details': {'reasoning_tokens': 27}, 'prompt_time': 0.03773901, 'prompt_tokens_details': None, 'queue_time': 0.315023068, 'total_time': 0.179367237}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4b2f03d631', 'service_tier': 'on_demand', 'finish_reason': 'tool_call

In [ ]:
llm_with_tools.invoke(messages).content

''